# Filtreler ve Net'ler

**Modüller:** `pytop.filters`, `pytop.nets`  
**Konu:** Genel topolojik uzaylarda yakınsama — dizilerin yetersiz kaldığı durumlarda filtreler ve net'ler kullanılır.

---

Metrik uzaylarda yakınsama dizilerle tanımlanır. Ancak genel topolojik uzaylarda diziler yetersizdir:
- Birinci sayılabilirlik aksiyomu sağlanmıyorsa dizi yakınsaması topolojiyi tanımlamaz.
- Kompaktlık, dizi kompaktlığından farklı olabilir.

**Filtreler** (Cartan, 1937) ve **net'ler** (Moore–Smith, 1922) bu boşluğu kapatır ve her topolojik uzayda tam bir yakınsama teorisi sağlar.

Bu notebook beş bölümden oluşur:
1. **Konu** — Filtre ve net tanımları, yakınsama ve küme noktası kavramları
2. **Teoremler** — Komşuluk filtresi, net-filtre eşdeğerliği, kompaktlık karakterizasyonu
3. **API** — `pytop.filters` ve `pytop.nets` fonksiyon referansı
4. **Örnekler** — Dört senaryo: filtre aksiyomlarından net yakınsamasına
5. **Alıştırmalar** — Kodlama ve teori

## 1. Konu

### 1.1 Filtreler

Bir küme $X$ üzerinde **filtre**, $\mathcal{F} \subseteq \mathcal{P}(X)$ ailesidir; şu koşulları sağlar:

- **(F1)** $\emptyset \notin \mathcal{F}$ (boş küme yok),
- **(F2)** $A, B \in \mathcal{F} \Rightarrow A \cap B \in \mathcal{F}$ (sonlu kesişimlere kapalı),
- **(F3)** $A \in \mathcal{F},\ A \subseteq B \subseteq X \Rightarrow B \in \mathcal{F}$ (yukarı kapalı).

**Filtre tabanı** $\mathcal{B}$: boş olmayan, her $B_1, B_2 \in \mathcal{B}$ için bir $B_3 \in \mathcal{B}$ ile $B_3 \subseteq B_1 \cap B_2$ olan aile.

**Yakınsama:** Topolojik uzay $(X, \tau)$'da filtre $\mathcal{F}$, $x \in X$'e yakınsıyorsa ($\mathcal{F} \to x$): $x$'in her açık komşuluğu $\mathcal{F}$'ye aittir.

**Küme noktası (cluster point):** $x$, $\mathcal{F}$'nin küme noktasıdır $\Leftrightarrow$ $x$'in her açık komşuluğu, $\mathcal{F}$'nin her elemanıyla kesişir.

### 1.2 Net'ler

**Yönlendirilmiş küme** $(I, \leq)$: Her $i, j \in I$ için $i \leq k$ ve $j \leq k$ olan bir $k$ vardır.

**Net:** $\nu: I \to X$ fonksiyonu; bir yönlendirilmiş kümeden uzaya gider. Diziler, $I = \mathbb{N}$ ile net'lerin özel halidir.

**Net yakınsaması:** $\nu \to x$ $\Leftrightarrow$ her açık komşuluk $U \ni x$ için $\exists i_0$ s.t. $i \geq i_0 \Rightarrow \nu(i) \in U$.

**Sonunda $A$'da (eventually in):** $\exists i_0 \in I: i \geq i_0 \Rightarrow \nu(i) \in A$.

**Sık sık $A$'da (frequently in):** $\forall i_0 \in I: \exists i \geq i_0$ ile $\nu(i) \in A$.

In [ ]:
from pytop.filters import (
    is_filter, is_filter_base, generated_filter,
    filter_converges_to, filter_cluster_points, filter_clusters_at,
    neighborhood_filter_base, analyze_filter, is_finer_filter,
)
from pytop.nets import (
    is_directed_set, net_converges_to,
    is_eventually_in, is_frequently_in,
    net_cluster_points, analyze_net,
)
from pytop.spaces import TopologicalSpace

# 4 noktalı uzay: {0,1,2,3}, açık kümeler: {}, {0,1,2,3}, {0,1}, {2,3}
X = TopologicalSpace(
    carrier=frozenset([0, 1, 2, 3]),
    topology=frozenset([
        frozenset(), frozenset([0, 1, 2, 3]),
        frozenset([0, 1]), frozenset([2, 3]),
    ])
)

# Filtre: {{0,1,2,3}, {0,1}} — (F1)-(F3) kontrol
F = [frozenset([0, 1, 2, 3]), frozenset([0, 1])]
r = is_filter(list(X.carrier), F)
print("Filtre mi:", r.value)
print("Gerekçe:", r.justification[0])

## 2. Teoremler

### Teorem 1 — Komşuluk Filtresi (Cartan)

$(X, \tau)$ topolojik uzay, $x \in X$. $x$'i içeren tüm açık kümelerin ailesi $\mathcal{N}(x)$ bir filtre tabanı oluşturur; üretilen filtre $\mathcal{F}(x)$ **komşuluk filtresidır**.

**Kanıt:** (F1) $x \in U$ olduğundan $U \neq \emptyset$. (F2) $U, V \in \mathcal{N}(x) \Rightarrow U \cap V$ açık, $x \in U \cap V$. (F3) Yukarı kapanma üretim işleminde elde edilir. $\square$

**Sonuç:** $\mathcal{F} \to x \Leftrightarrow \mathcal{F} \supseteq \mathcal{F}(x)$ (filtre inceliği sıralamasına göre $\mathcal{F}$, komşuluk filtresinden daha ince).

### Teorem 2 — Net ve Filtre Eşdeğerliği

Her net, kanonik bir filtre belirler; her filtre de bir net belirler. Özellikle:
$$\nu \to x \Longleftrightarrow \mathcal{F}_\nu \to x,$$
burada $\mathcal{F}_\nu = \{A \subseteq X : \nu \text{ sonunda } A\text{'da}\}$.

**Kanıt:** $\Rightarrow$: $\nu \to x$ ise her komşuluk $U \ni x$ için $\nu$ sonunda $U$'da, dolayısıyla $U \in \mathcal{F}_\nu$, yani $\mathcal{F}_\nu \supseteq \mathcal{F}(x)$. $\Leftarrow$: Her komşuluk $U \in \mathcal{F}_\nu$ olduğundan $\nu$ sonunda $U$'da. $\square$

### Teorem 3 — Kompaktlık Karakterizasyonu

$(X, \tau)$ kompakt $\Leftrightarrow$ $X$ üzerindeki her filtrenin en az bir küme noktası vardır.

**Kanıt ($\Rightarrow$):** Sonlu kesişim özelliği: Filtre elemanları kapalı süperkümeler alındığında, kompaktlık sayesinde kesişim boş değil; bu bir küme noktası verir. $\Leftarrow$: Her açık örtünün sonlu alt örtüsü olmadığı varsayımı, küme noktasız bir filtre inşa eder; çelişki. $\square$

In [ ]:
# Teorem 1: Komşuluk filtresi tabanı
nbhd = neighborhood_filter_base(X, 0)
print("0'ın komşuluk taban elemanları:", nbhd.value)

# Bu tabandan üretilen filtrenin 0'a yakınsadığını doğrula
F0 = list(generated_filter(list(X.carrier), [frozenset([0, 1])]).value)
conv = filter_converges_to(X, F0, 0)
print("F0 -> 0:", conv.value)

# Teorem 3: Küme noktaları
cluster = filter_cluster_points(X, F0)
print("F0'ın küme noktaları:", cluster.value)

## 3. API Referansı

### `pytop.filters`

| Fonksiyon | İmza | Açıklama |
|-----------|------|----------|
| `is_filter` | `(carrier, family) -> Result` | (F1)–(F3) aksiyomlarını kontrol eder |
| `is_filter_base` | `(carrier, base) -> Result` | Filtre tabanı koşullarını denetler |
| `generated_filter` | `(carrier, base) -> Result` | Tabandan filtre üretir (yukarı kapanma) |
| `neighborhood_filter_base` | `(space, point) -> Result` | Noktanın komşuluk taban filtresi |
| `filter_converges_to` | `(space, family, point) -> Result` | Filtre yakınsamasını kontrol eder |
| `filter_cluster_points` | `(space, family) -> Result` | Tüm küme noktalarını bulur |
| `filter_clusters_at` | `(space, family, point) -> Result` | Tek noktada küme kontrolü |
| `is_finer_filter` | `(carrier, F1, F2) -> Result` | $F_1 \supseteq F_2$ (F1 daha ince) |
| `analyze_filter` | `(space, family) -> Result` | Tüm filtre diagnostiklerini bir arada |

### `pytop.nets`

| Fonksiyon | İmza | Açıklama |
|-----------|------|----------|
| `is_directed_set` | `(index_set, preorder) -> Result` | Yönlendirilmiş küme kontrolü |
| `net_converges_to` | `(space, idx, preorder, net_fn, point) -> Result` | Net yakınsaması |
| `is_eventually_in` | `(idx, preorder, net_fn, subset) -> Result` | Sonunda $A$'da |
| `is_frequently_in` | `(idx, preorder, net_fn, subset) -> Result` | Sık sık $A$'da |
| `net_cluster_points` | `(space, idx, preorder, net_fn) -> Result` | Net küme noktaları |
| `analyze_net` | `(space, idx, preorder, net_fn) -> Result` | Tam net diagnostiği |

In [ ]:
# API özeti: analyze_filter
F_valid = [frozenset([0, 1]), frozenset([0, 1, 2, 3])]
report = analyze_filter(X, F_valid)
print("=== analyze_filter ===")
print("Filtre geçerli mi:", report.value['filter_check'].value)
print("Küme noktaları:", report.value['cluster_points'].value)

# analyze_net
I = [0, 1, 2, 3]
pre = {(i, j) for i in I for j in I if i <= j}
def nu(k): return 0 if k < 2 else 1

net_report = analyze_net(X, I, pre, nu)
print("\n=== analyze_net ===")
print("Net küme noktaları:", net_report.value['cluster_points']['value'])

## 4. Örnekler

### Örnek 1 — Filtre Aksiyomlarının Doğrulanması

İki farklı aile incelenecek: biri gerçek filtre, diğeri (F3)'ü ihlal ediyor.

In [ ]:
carrier = [1, 2, 3, 4]

# Geçerli filtre: {1,2,3,4} ve {1,2} — yukarı kapalı değil, (F3) başarısız olabilir
# Gerçek filtre: tüm {1,2} üstkümelerini al
F_true = [
    frozenset([1, 2]),
    frozenset([1, 2, 3]),
    frozenset([1, 2, 4]),
    frozenset([1, 2, 3, 4]),
]
r1 = is_filter(carrier, F_true)
print("F_true filtre mi:", r1.value)

# Geçersiz: {1,2} var ama {1,2,3} yok — (F3) ihlali
F_bad = [frozenset([1, 2, 3, 4]), frozenset([1, 2])]
r2 = is_filter(carrier, F_bad)
print("F_bad filtre mi:", r2.value)
print("Neden:", r2.justification[0])

# is_filter_base: {1,2} tek başına filtre tabanı mı?
r3 = is_filter_base(carrier, [frozenset([1, 2])])
print("\n{1,2} filtre tabanı mı:", r3.value)

### Örnek 2 — Filtre İnceliği (Finer Filter)

Filtreler kısmen sıralanır: $\mathcal{F}_1 \supseteq \mathcal{F}_2$ ise $\mathcal{F}_1$, $\mathcal{F}_2$'den **daha ince** (finer). Daha ince filtreler daha fazla bilgi taşır; bir filtrenin en ince uzantısı **ultrafiltredir**.

In [ ]:
carrier = [1, 2, 3, 4]

# Kaba filtre: yalnızca tam küme
F_coarse = [frozenset([1, 2, 3, 4])]

# İnce filtre: {1,2,3,4} ve {1,2} ve {1,2,3} ve {1,2,4}
F_fine = [
    frozenset([1, 2]),
    frozenset([1, 2, 3]),
    frozenset([1, 2, 4]),
    frozenset([1, 2, 3, 4]),
]

r = is_finer_filter(carrier, F_fine, F_coarse)
print("F_fine, F_coarse'dan daha ince mi:", r.value)
print("Gerekçe:", r.justification[0])

# Ters yön: F_coarse, F_fine'dan daha ince mi?
r2 = is_finer_filter(carrier, F_coarse, F_fine)
print("\nF_coarse, F_fine'dan daha ince mi:", r2.value)

### Örnek 3 — Net ile Yakınsama

Sıradan dizi: $\nu(n) = n \bmod 2$ (0,1,0,1,...) — İkili uzayda hangi noktaya yakınsar?

İndirge uzayı: $X = \{0,1\}$, topoloji: $\{\emptyset, \{0,1\}, \{0\}, \{1\}\}$ (ayrık).

In [ ]:
X_discrete = TopologicalSpace(
    carrier=frozenset([0, 1]),
    topology=frozenset([
        frozenset(), frozenset([0, 1]),
        frozenset([0]), frozenset([1]),
    ])
)

I = list(range(8))  # indeks kümesi
pre = {(i, j) for i in I for j in I if i <= j}

def alternating_net(k): return k % 2  # 0,1,0,1,...

# Yakınsama kontrolü
r0 = net_converges_to(X_discrete, I, pre, alternating_net, 0)
r1 = net_converges_to(X_discrete, I, pre, alternating_net, 1)
print("Degisimli net -> 0:", r0.value, "|", r0.justification[0][:60])
print("Degisimli net -> 1:", r1.value)

# Küme noktaları
cp = net_cluster_points(X_discrete, I, pre, alternating_net)
print("Kume noktaları:", sorted(cp.value))

# eventually / frequently
in0 = is_eventually_in(I, pre, alternating_net, frozenset([0]))
freq0 = is_frequently_in(I, pre, alternating_net, frozenset([0]))
print("\nSonunda {0}'da:", in0.value, "| Sik sik {0}'da:", freq0.value)

### Örnek 4 — Karşılaştırma Tablosu: Dizi vs. Net vs. Filtre

Üç mekanizma karşılaştırılıyor. Her biri aynı topolojik olgu için eşdeğerdir, ancak farklı uzaylarda farklı etkinlik gösterir:

| Özellik | Dizi | Net | Filtre |
|---------|------|-----|--------|
| İndeks kümesi | $\mathbb{N}$ | Yönlendirilmiş küme | — |
| Kapalılık karakterizasyonu | 1. sayılabilir uzaylarda | Her uzayda | Her uzayda |
| Kompaktlık | Dizi-kompaktlık (zayıf) | Net kompaktlık | Ultrafiltre kompaktlığı |
| Küme noktası | Alt-dizi limiti | Küme noktası | Ultrafiltre uzantısı |
| Teorik güç | Kısıtlı | Tam | Tam |
| `pytop` API | `pytop.sequences` | `pytop.nets` | `pytop.filters` |

In [ ]:
# Net ve filtre yakınsamasının eşdeğerliği: sabit net
X4 = TopologicalSpace(
    carrier=frozenset([0, 1, 2]),
    topology=frozenset([
        frozenset(), frozenset([0, 1, 2]),
        frozenset([0]), frozenset([1, 2]),
    ])
)

I = [0, 1, 2]
pre = {(i, j) for i in I for j in I if i <= j}

for target in [1, 2]:
    def const_net(k, t=target): return t
    r_net = net_converges_to(X4, I, pre, const_net, target)
    # Karsilestirici filtre: komşuluk filtresi
    base_sets = list(neighborhood_filter_base(X4, target).value)
    F = list(generated_filter(list(X4.carrier), base_sets).value)
    r_fil = filter_converges_to(X4, F, target)
    print(f"Sabit net -> {target}: Net={r_net.value}, Filter={r_fil.value}")

## 5. Alıştırmalar

**Alıştırma 1.** `carrier = [1,2,3,4,5]` üzerinde bir filtre `F` oluşturun ve `is_filter` ile doğrulayın. Sonra `is_finer_filter` ile $F$'nin $\{\{1,2,3,4,5\}\}$ filtresinden daha ince olduğunu gösterin.

**Alıştırma 2.** Aşağıdaki 5 noktalı uzayda 2. noktanın komşuluk filtresi tabanını hesaplayın:
```
carrier = {1,2,3,4,5}
topology = {∅, X, {1,2}, {3,4,5}, {1,2,3,4,5}}
```
Bu filtrenin küme noktalarını bulun. Hangi noktaya yakınsar?

**Alıştırma 3.** Net $\nu(k) = k \bmod 3$ (0,1,2,0,1,2,...) ile $I = \{0,\ldots,11\}$, ayrık topolojili $\{0,1,2\}$ uzayında deneyin. `is_eventually_in`, `is_frequently_in`, `net_cluster_points` sonuçlarını yorumlayın.

**Alıştırma 4.** İndirgeme toposu (Sierpinski uzayı) $\{0,1\}$, açık: $\{\emptyset, \{1\}, \{0,1\}\}$'de şu iki filtreyi inceleyin:
- $\mathcal{F}_a = \{\{1\}, \{0,1\}\}$ — nereye yakınsar?
- $\mathcal{F}_b = \{\{0,1\}\}$ — nereye yakınsar?

`filter_converges_to` ve `filter_cluster_points` kullanın.

In [ ]:
# Alıştırma 4 kurulumu — Sierpinski uzayı
S = TopologicalSpace(
    carrier=frozenset([0, 1]),
    topology=frozenset([
        frozenset(),
        frozenset([1]),
        frozenset([0, 1]),
    ])
)

Fa = [frozenset([1]), frozenset([0, 1])]
Fb = [frozenset([0, 1])]

print("Fa -> 0:", filter_converges_to(S, Fa, 0).value)
print("Fa -> 1:", filter_converges_to(S, Fa, 1).value)
print("Fb -> 0:", filter_converges_to(S, Fb, 0).value)
print("Fb -> 1:", filter_converges_to(S, Fb, 1).value)
print("Fa kume noktalari:", filter_cluster_points(S, Fa).value)
print("Fb kume noktalari:", filter_cluster_points(S, Fb).value)

---
## Özet

| Kavram | Açıklama |
|--------|----------|
| Filtre | `∅ ∉ F`; sonlu kesişimlere kapalı; yukarı kapalı — (F1)-(F3) |
| Filtre tabanı | Her iki eleman için ortak bir alt eleman var |
| Komşuluk filtresi | Bir noktanın tüm açık komşuluklarının filtresi |
| Filtre yakınsaması | `F → x` iff `F ⊇ F(x)` (komşuluk filtresinden daha ince) |
| Filtre küme noktası | Her komşuluk her filtre elemanıyla kesişir |
| Filtre inceliği | `F₁ ⊇ F₂` ⟹ F₁ daha ince (finer); daha fazla bilgi taşır |
| Net | `ν: I → X`; `I` yönlendirilmiş küme; dizinin genellemesi |
| Net yakınsaması | Her komşuluk `U` için sonunda `ν(i) ∈ U` |
| `is_eventually_in` | `∃ i₀: i ≥ i₀ ⇒ ν(i) ∈ A` |
| `is_frequently_in` | `∀ i₀ ∃ i ≥ i₀: ν(i) ∈ A` |
| Net–Filtre eşdeğerliği | `ν → x ⟺ {A : ν sonunda A'da} → x` |
| Kompaktlık | Her filtrenin küme noktası var ⟺ kompakt |
| `analyze_filter` | Filtre diagnostiklerini toplu döner |
| `analyze_net` | Net diagnostiklerini toplu döner |

**Sonraki konu:** `pytop.persistent_homology` — filtrasyonlar, kalıcılık, barkodlar